# [1교시]

In [ ]:
# 주성분 분석 => 평균 제거 => PCA 적용 => 변환된 데이터
# 각 Feature의 평균을 빼서 데이터의 중심을 원점으로 옮기기
import matplotlib.pyplot as plt
import numpy as np
ax = plt.subplot(1,1,1,aspect ='equal')
x = np.array([[2, 4], [4, 6], [6, 8]])
ax.scatter(x[:,0], x[:,1])
x -= x.mean(axis=0).astype(int) # 각 열의 평균 계산
ax.scatter(x[:,0], x[:,1])
# 평균을 제거하지 않으면 데이터가 원점에서 멀리 떨어져 있어 PCA가 제대로 작동하지 않을 수 있음

In [ ]:
import numpy as np
np.random.seed(42)
m = 60
# 3차원데이터
w1, w2 = 0.1,0.3
noise = 0.1

angle = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m,3))

# 서로 약간 상관관계 있는 구조
X[:,0] = np.cos(angle) + np.sin(angle)/2 + noise*np.random.rand(m)
X[:,1] = np.sin(angle) * 0.7 + noise * np.random.rand(m)
X[:,2] = X[:,0] * w1 + X[:,1]*w2 + noise*np.random.rand(m)

# PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

# 변환전 (X의 일부 차원만 시각화)
plt.subplot(1,2,1)
plt.scatter(X[:,0],X[:,1])
plt.title('original data')

plt.subplot(1,2,2)
plt.scatter(X2D[:,0],X2D[:,1])
plt.title('PCA data')

plt.show()

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
# 타원형 데이터 생성
m = 200
# 길게 늘어진 직선형태
X = np.random.randn(m,2)
X[:,0]*=3
# 회전(기울기)
theta = np.pi /4 # 45도 회전
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta), np.cos(theta)]
])
X = X@rotation_matrix
#  PCA 적용
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

fig, ax = plt.subplots(1,2,figsize=(10,4))

ax[0].scatter(X[:,0], X[:,1])
ax[0].set_title('original')

ax[1].scatter(X2D[:,0], X2D[:,1])
ax[1].set_title('PCA')

plt.tight_layout()
plt.show()

# [2교시]

In [ ]:
# wine 데이터셋
from sklearn.datasets import load_wine
wine = load_wine()

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# 데이터 로드
wine = load_wine()
X_wine = wine.data  # shape (178, 13)

# 표준화 (PCA 전에 권장)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_wine)

# PCA로 2차원으로 축소
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)  # shape (178, 2)

# 원래 PCA 결과 복사 (비교용)
X_orig = X_pca.copy()

# x축을 3배로 늘려 타원형으로 만듦 (원래 코드의 의도 재현)
X_mod = X_pca.copy()
X_mod[:, 0] *= 3

# 45도 회전 (시계 반대 방향: +45도)
theta = np.pi / 4
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])
X_mod_rot = X_mod @ rotation_matrix

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_orig[:, 0], X_orig[:, 1], c='C0', alpha=0.7)
axes[0].set_title('PCA (target0)')

axes[1].scatter(X_mod[:, 0], X_mod[:, 1], c='C1', alpha=0.7)
axes[1].set_title('PCA (target1)')

axes[2].scatter(X_mod_rot[:, 0], X_mod_rot[:, 1], c='C2', alpha=0.7)
axes[2].set_title('PCA (target2)')

plt.tight_layout()
plt.show()

# 설명 분산 비율 출력
print(("설명 분산 비율"), pca.explained_variance_ratio_)

=========================

In [ ]:
# wine 데이터셋
from sklearn.datasets import load_wine
wine = load_wine()

# 데이터 + 모델 조합, 성능 측정
# 데이터(PCA변환) + 모델 조합, 성능 측정
X = wine.data
y = wine.target

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('pca', PCA(n_components=2)),
])
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold

models = {'sgd' : SGDClassifier(random_state=42),
          'randomF' : RandomForestClassifier(random_state=42),
          'tree' : DecisionTreeClassifier(random_state=42),
         }

for model_name, model in models.items():
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    fold = StratifiedKFold(shuffle=True, random_state=42)
    y_predict = cross_val_predict(model, X_scaled, y, cv=fold)
    print(model_name)
    print(classification_report(y, y_predict))

=================================

# [3교시]

In [2]:
# votingclass 다수결
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
voting = VotingClassifier([
    ('sgd', SGDClassifier(random_state=42)),
    ('randomF', RandomForestClassifier(random_state=42)),
    ('tree', DecisionTreeClassifier(random_state=42))
], voting = 'hard')
y_predict = cross_val_predict(voting, X_scaled, y, cv=fold)
print(classification_report(y, y_predict))

NameError: name 'SGDClassifier' is not defined

In [3]:
# votingclass 다수결
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
voting = VotingClassifier([
    ('sgd', SGDClassifier(random_state=42, loss='log_loss')),
    ('randomF', RandomForestClassifier(random_state=42)),
    ('svc', SVC(random_state=42, probability=True, kernel='rbf'))
], voting = 'soft')
y_predict = cross_val_predict(voting, X_scaled, y, cv=fold)
print(classification_report(y, y_predict))

# verbose = 1 : baggingclassifier안에 넣어서 혼자 중얼거리며 실행이 된다.
bagging = BaggingClassifier(voting, n_estimators=10, random_state=42)
y_predict = cross_val_predict(bagging, X_scaled, y, cv=fold)
print(classification_report(y, y_predict))

stacking = StackingClassifier([
    ('sgd', SGDClassifier(random_state=42, loss='log_loss')),
    ('randomF', RandomForestClassifier(random_state=42)),
    ('svc', SVC(random_state=42, probability=True, kernel='rbf'))
], final_estimator=RandomForestClassifier(random_state=42))
y_predict = cross_val_predict(stacking, X_scaled, y, cv=fold)
print(classification_report(y, y_predict))

NameError: name 'SGDClassifier' is not defined

In [ ]:
pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('pca', PCA(n_components=2)),
])
x_pipeline = pipeline.fit_transform(X)
stacking = StackingClassifier([
    ('sgd', SGDClassifier(random_state=42, loss='log_loss')),
    ('randomF', RandomForestClassifier(random_state=42)),
    ('svc', SVC(random_state=42, probability=True, kernel='rbf'))
], final_estimator=RandomForestClassifier(random_state=42))
y_predict = cross_val_predict(stacking, x_pipeline, y, cv=fold)
print(classification_report(y, y_predict))

In [ ]:
# n_compnents 찾기, 누적 분산
import numpy as np
pca = PCA(n_components=13)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_pca = pca.fit_transform(X_scaled)
cumsum_variable = np.cumsum(
    pca.explained_variance_ratio_
)
print(cumsum_variable) # 95%이상이면 좋음, 90%이상이면 빠르게, 99%이상이면 정보보존을 우선

np.argmax(cumsum_variable >= 0.95) + 1

In [ ]:
# 시각적 엘보우... 급격히 증가후 완만해지는 구간
import matplotlib.pyplot as plt
plt.plot(cumsum_variable)
plt.axhline(y=0.95, linestyle='--', c='red')

In [ ]:
# 자동으로 찾기
# n_components 찾기, 누적분산
import numpy as np
pca = PCA(n_components=0.95)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_pca = pca.fit_transform(X_scaled)
print(pca.n_components_)


# [4교시]

In [ ]:
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 데이터 로드
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
target_names = wine.target_names

# 원본 데이터 (첫 두 특성) 시각화
plt.figure(figsize=(6,5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolor='k', s=40)
plt.title('Original data (feature 0 vs 1)')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.show()

# PCA 적용 (2차원)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# PCA 결과 시각화 (클래스별 색상)
plt.figure(figsize=(6,5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k', s=40)
plt.title('PCA (2 components)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

# 설명된 분산 비율 출력
print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Sum explained variance:', pca.explained_variance_ratio_.sum())


# [5교시]

In [5]:
from glob import glob
file_path = r'C:\python-src\Daily\day24\data\*.csv'
files = glob(file_path)
files

['C:\\python-src\\Daily\\day24\\data\\layout_info.csv',
 'C:\\python-src\\Daily\\day24\\data\\sample_submission.csv',
 'C:\\python-src\\Daily\\day24\\data\\test.csv',
 'C:\\python-src\\Daily\\day24\\data\\train.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_강원_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_경기_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_경남_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_경북_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_광주_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_대구_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_대전_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_부산_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_서울_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_세종_202512.csv',
 'C:\\python-src\\Daily\\day24\\data\\소상공인시

In [6]:
import pandas as pd
df = pd.read_csv(files[0])
# 불필요한 컬럼제거 - 의미가 중복된 데이터는 제거하는 것이 좋다.

In [7]:
new_cols = ['상호명', '지점명', '상권업종대분류명', '상권업종중분류명', '상권업종소분류명', 
            '시도명', '시군구명', '행정동명', '법정동명', '도로명주소', '위도', '경도']
df[new_cols].head()

KeyError: "None of [Index(['상호명', '지점명', '상권업종대분류명', '상권업종중분류명', '상권업종소분류명', '시도명', '시군구명', '행정동명',\n       '법정동명', '도로명주소', '위도', '경도'],\n      dtype='object')] are in the [columns]"

In [8]:
# 결측치 확인
shop = df[new_cols]
shop.isnull().sum()

KeyError: "None of [Index(['상호명', '지점명', '상권업종대분류명', '상권업종중분류명', '상권업종소분류명', '시도명', '시군구명', '행정동명',\n       '법정동명', '도로명주소', '위도', '경도'],\n      dtype='object')] are in the [columns]"

In [9]:
%pip install missingno

Note: you may need to restart the kernel to use updated packages.


In [10]:
# 한글설정
import matplotlib.pyplot as plt
plt.rc('font', family='Malgun Gothic')

In [11]:
# 결측치 시각적으로 보기
import missingno as msno
msno.matrix(shop)

NameError: name 'shop' is not defined

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
plt.plot(shop['경도'],shop['위도'], 'o', markersize=1)
plt.xlabel('위도')
plt.ylabel('경도')
plt.title('매장 위치 분포')
plt.show()

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(12,6))
sns.scatterplot(data=shop,x='경도',y='위도',hue='시군구명',ax=ax)

In [12]:
# 상권업종 대분류명이  교육과 관련된 정보
import numpy as np
edu_shop = shop[shop['상권업종대분류명']=='교육']
sns.scatterplot(edu_shop,x='경도',y='위도')

NameError: name 'shop' is not defined

# [6교시]

In [13]:
# shop = df[cols].copy()

# '교육' 포함 필터, 위도/경도 숫자 변환 및 결측 제거
edu = shop[shop['상권업종대분류명'].astype(str).str.contains('교육', na=False)]
edu['위도'] = pd.to_numeric(edu['위도'], errors='coerce')
edu['경도'] = pd.to_numeric(edu['경도'], errors='coerce')
edu = edu.dropna(subset=['위도','경도'])

# 시군구가 많을 때: 상위 N개만 색상, 나머지는 '기타'
N = 12
topN = edu['시군구명'].value_counts().nlargest(N).index
edu['시군구_top'] = edu['시군구명'].where(edu['시군구명'].isin(topN), '기타')

plt.figure(figsize=(12,8))
ax = sns.scatterplot(data=edu, x='경도', y='위도',
                     hue='시군구_top', palette='tab10',
                     alpha=0.7, s=40)
ax.set_title('교육 관련 상권 분포 (시군구별)')
ax.set_xlabel('경도'); ax.set_ylabel('위도')
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left', title='시군구')
plt.tight_layout()
plt.show()

NameError: name 'shop' is not defined

In [14]:
# 반경 200m 내 점포수
# 동일 업종수
# 경쟁 강도(같은 업종 density)

# step1 결측치 제거
# step2 공간 Feature 생성  KDTree
# step3 모델 정의(분류 회귀)
# step4 학습
from sklearn.neighbors import BallTree
coors = np.radians(shop[['위도','경도']].values)  # 각도를 degree ~~도   -> 라디안 radian으로 변환
tree = BallTree(coors,metric='haversine')  # 단위가 radian

# 500m반경
radius = 0.5 / 6371
counts = tree.query_radius(coors,r=radius,count_only=True)
shop.loc[:,'store_density_500m']  = counts

NameError: name 'shop' is not defined

In [15]:
# 음식점 기준
target_category = '음식'
food_mask = (shop['상권업종대분류명'] == target_category)

food_coors = np.radians(shop.loc[food_mask,['위도','경도']].values)
food_tree = BallTree(food_coors,metric='haversine') 

# 전체 좌표 기준으로 음식점 수 계산
food_counts = food_tree.query_radius(coors, r=radius, count_only=True)
shop.loc[:,'fodd_density_500'] = food_counts

NameError: name 'shop' is not defined

In [ ]:
# 경쟁강도
shop.loc[:,'competition_ratio'] = (shop['fodd_density_500'] / shop['store_density_500m'])
# 0에 가까울수록 경쟁적음  1에 가까울수록 경쟁이 심함

In [ ]:
indices = tree.query_radius(coors, r=radius)
diversity = [len(shop.iloc[idx_list]['상권업종중분류명'].unique()) for idx_list in indices]

In [ ]:
shop['category_deiversity_500m'] = diversity

In [ ]:
shop.loc[:,'위도':].head(2)

In [ ]:
X = shop.loc[:,'위도':]
y = (
    0.4*X['store_density_500m']  # 전체상권 활성도
    -0.3*X['competition_ratio']  # 경쟁강도(패널티)
    +0.3*X['category_deiversity_500m']  #상권의 다양성
)

In [ ]:
# 주변 점포수 많음 - >사람 많음 -> 좋은 입지
X.head()

# [7교시]

In [ ]:
from lightgbm import LGBMRegressor
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)
model.fit(X,y)

In [ ]:
# 기존 매장위치가 아니라 새로운 후보 위치를 만들어서 평가
# 새로 어디에 창업할지에 대한 정보가 없음
# 가상의 후보 위치 생성
########## 데이터가 존재하는 곳의 지역범위 추출
lat_min, lat_max =  shop['위도'].min(), shop['위도'].max()
lon_min, lon_max = shop['경도'].min(), shop['경도'].max()
################### 격자 생성
lat_grid = np.linspace(lat_min,lat_max, 50) #50
lon_grid = np.linspace(lon_min,lon_max, 50) #50 
# 2500 후보 위치
# 2D grid 생성  위도 경도 모든 조합 생성
grid = np.array(np.meshgrid(lat_grid,lon_grid)).T.reshape(-1,2)
# lat : [a,b]  long: [x,y]
#  [a,x] [a,y] [b,x] [b,y]
# dataframe 변환
grid_df = pd.DataFrame(grid, columns=['위도','경도'])
# 라디안 변환
grid_coords = np.radians(grid)

# 주변상권분석(주변 500m 점포가 몇개?)
grid_store_counts = tree.query_radius(grid_coords,r=radius,count_only=True)
grid_df['store_density_500m'] = grid_store_counts
# 이 위치 주변에 음식점이 몇개 있냐?
grid_food_counts = food_tree.query_radius(grid_coords,r=radius,count_only=True)
grid_df['food_density_500m'] = grid_food_counts

grid_df['competition_ratio'] = (
    grid_df['food_density_500m'] / grid_df['store_density_500m']
)

#다양성
grid_indicies = tree.query_radius(grid_coords,r=radius)

grid_diversity = [   shop.iloc[idx_list]['상권업종중분류명'].unique() 
                  for idx_list in grid_indicies ]

grid_df.loc[:,'category_deiversity_500m'] = [len(array) for array in grid_diversity]

# 예측 및 추천
grid_x = grid_df.copy()
grid_df['pred_score'] = model.predict(grid_x)

In [ ]:
# top 10 추천
top_k = grid_df.sort_values('pred_score', ascending=False).head(10)
top_k

In [ ]:
sns.scatterplot(data=top_k, x='경도', y='위도')
plt.show()

In [ ]:
%pip install folium

In [ ]:
!pip install -U ipywidgets folium

In [ ]:
from IPython.display import display
import folium
# 지도의 중심
center_lat = shop['위도'].mean()
center_lon = shop['경도'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13)
for idx, row in top_k.iterrows():
    folium.Marker(
    location = [row['위도'], row['경도']],
    popup = f"score: {row['pred_score']:.2f}",
    icon = folium.Icon(color='red'),
    ).add_to(m)
display(m)

In [ ]:
import folium
# 지도의 중심
center_lat =  shop['위도'].mean()
center_lon =  shop['경도'].mean()
m = folium.Map(location=[center_lat, center_lon],zoom_start=13)
for idx, row in top_k.iterrows():
    folium.Marker(
        location = [ row['위도']  , row['경도']  ],
        popup = f"score:{row['pred_score']:.2f}",
        icon = folium.Icon(color='red'),
    ).add_to(m)
m.save('map.html')

In [ ]:
from glob import glob
import pandas as pd

file_path = r'C:\python-src\Daily\day24\data\*.csv'
files = glob(file_path)          # 패턴 전체로 glob 실행
print(len(files))                # 파일 개수 확인
print(files)                     # 파일 목록 확인

# 앞 4개를 제외하고 9번째(인덱 8)를 읽고 싶다면
subset = files[4:]
if len(subset) > 8:
    shop = pd.read_csv(subset[8])
else:
    raise IndexError(f"subset 길이 {len(subset)}: 인덱스 8에 접근 불가")




21
['C:\\python-src\\Daily\\day24\\data\\layout_info.csv', 'C:\\python-src\\Daily\\day24\\data\\sample_submission.csv', 'C:\\python-src\\Daily\\day24\\data\\test.csv', 'C:\\python-src\\Daily\\day24\\data\\train.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_강원_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_경기_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_경남_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_경북_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_광주_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_대구_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_대전_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_부산_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_서울_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권)정보_세종_202512.csv', 'C:\\python-src\\Daily\\day24\\data\\소상공인시장진흥공단_상가(상권

In [ ]:
import pandas as pd
shop = pd.read_csv(files[12])
db = shop.copy()
db

In [106]:
# 상권업종중분류명에 어떤 값들이 있는지 확인
print(shop['상권업종소분류명'].unique())

['치킨' '고용 알선업' '카페' '노래방' '시계/귀금속 소매업' '부동산 중개/대리업' '기타 의류 소매업' '요리 주점'
 '가구 소매업' '여성 의류 소매업' '전시/컨벤션/행사 대행 서비스업' '헬스장' '액세서리/잡화 소매업' '돼지고기 구이/찜'
 '약국' '경영 컨설팅업' '자동차 세차장' '문구/회화용품 소매업' '유아용 의류 소매업' '건설/건축자재 소매업'
 '핸드폰 소매업' '세무사' '치과의원' '핸드폰/통신장비 수리업' '기타 여행 보조/예약 서비스업' '담배/전자담배 소매업'
 '컴퓨터/소프트웨어 소매업' '기념품점' '주유소' '미용실' '네일숍' '백반/한정식' '펜션' '변호사' '횟집'
 '모터사이클 및 부품 소매업' '신발 소매업' '중국집' '건축 설계 및 관련 서비스업' '입시·교과학원'
 '컴퓨터/사무기기 대여업' '피부/비뇨기과 의원' '스포츠/레크리에이션 용품 대여업' '세탁소' '편의점' '시각 디자인업'
 '일식 회/초밥' '사업/무형 재산권 중개업' '음악학원' '여관/모텔' '곡물/곡분 소매업' '태권도/무술학원'
 '요가/필라테스 학원' '내과/소아과 의원' '번역/통역 서비스업' '한방병원' '채소/과일 소매업' '서점' '일반병원'
 '미술학원' '그 외 기타 숙박업' '수산물 소매업' '인테리어 디자인업' '중고 상품 소매업' '교육컨설팅업' '정육점'
 '가전제품 소매업' '사업시설 유지·관리 서비스업' '음반/비디오물 소매업' '전문자격/고시학원' '자동차 정비소'
 '그 외 기타 간이 음식점' '여행사' '건축물 일반 청소업' '기타 기술/직업 훈련학원' '슈퍼마켓' '국/탕/찌개류'
 '해산물 구이/찜' '사진촬영업' '전기용품/조명장치 소매업' '명함/간판/광고물 제작' '사무기기 소매업' '광고 매체 판매업'
 '복권 발행/판매업' '청소년 수련시설' '사회교육시설' '마사지/안마' '직원 훈련기관' '예술품 소매업' '떡/한과'
 '남성 의류 소매업' '건강보조식품 소매업' '기타 엔지니

In [108]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from lightgbm import LGBMRegressor

# ---------------------------
# 0. 데이터 준비
# ---------------------------
# shop: 기존 데이터프레임
shop = shop.dropna(subset=['위도','경도','상권업종소분류명']).reset_index(drop=True)

# 좌표 → 라디안 변환
coords = np.radians(shop[['위도','경도']].values)

# BallTree 생성
tree = BallTree(coords, metric='haversine')

# 반경 설정 (500m)
radius = 0.5 / 6371


# ---------------------------
# 1. 전체 점포 밀집도
# ---------------------------
store_counts = tree.query_radius(coords, r=radius, count_only=True)
shop['store_density_500m'] = store_counts


# ---------------------------
# 2. 동일 업종 밀집도 (예: 카페)
# ---------------------------
target_category = '카페'

cafe_mask = (shop['상권업종소분류명'] == target_category)
cafe_coords = np.radians(shop.loc[cafe_mask, ['위도','경도']].values)

cafe_tree = BallTree(cafe_coords, metric='haversine')

cafe_counts = cafe_tree.query_radius(coords, r=radius, count_only=True)
shop['cafe_density_500m'] = cafe_counts


# ---------------------------
# 3. 경쟁 강도
# ---------------------------
shop['competition_ratio'] = (
    shop['cafe_density_500m'] / shop['store_density_500m']
).replace([np.inf, -np.inf], 0).fillna(0)


# ---------------------------
# 4. 상권 다양성
# ---------------------------
indices = tree.query_radius(coords, r=radius)

diversity = []
for idx_list in indices:
    categories = shop.iloc[idx_list]['상권업종소분류명']
    diversity.append(categories.nunique())

shop['category_diversity_500m'] = diversity


# ---------------------------
# 5. Feature 정리
# ---------------------------
features = [
    'store_density_500m',
    'cafe_density_500m',
    'competition_ratio',
    'category_diversity_500m',
    '위도',
    '경도'
]

X = shop[features]


# ---------------------------
# 6. Score 생성 (라벨 대신)
# ---------------------------
shop['score'] = (
    0.4 * shop['store_density_500m']
    - 0.3 * shop['competition_ratio']
    + 0.3 * shop['category_diversity_500m']
)

y = shop['score']


# ---------------------------
# 7. 모델 학습
# ---------------------------
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

model.fit(X, y)


# ---------------------------
# 8. 후보 위치 (Grid 생성)
# ---------------------------
lat_min, lat_max = shop['위도'].min(), shop['위도'].max()
lon_min, lon_max = shop['경도'].min(), shop['경도'].max()

lat_grid = np.linspace(lat_min, lat_max, 50)
lon_grid = np.linspace(lon_min, lon_max, 50)

grid = np.array(np.meshgrid(lat_grid, lon_grid)).T.reshape(-1, 2)

grid_df = pd.DataFrame(grid, columns=['위도','경도'])

grid_coords = np.radians(grid)


# ---------------------------
# 9. Grid Feature 생성
# ---------------------------
# 전체 밀집도
grid_store_counts = tree.query_radius(grid_coords, r=radius, count_only=True)

# 카페 밀집도
grid_cafe_counts = cafe_tree.query_radius(grid_coords, r=radius, count_only=True)

grid_df['store_density_500m'] = grid_store_counts
grid_df['cafe_density_500m'] = grid_cafe_counts

# 경쟁 강도
grid_df['competition_ratio'] = (
    grid_df['cafe_density_500m'] / grid_df['store_density_500m']
).replace([np.inf, -np.inf], 0).fillna(0)

# 다양성 (주의: 느림)
grid_indices = tree.query_radius(grid_coords, r=radius)

grid_diversity = []
for idx_list in grid_indices:
    categories = shop.iloc[idx_list]['상권업종소분류명']
    grid_diversity.append(categories.nunique())

grid_df['category_diversity_500m'] = grid_diversity


# ---------------------------
# 10. 예측 및 추천
# ---------------------------
grid_X = grid_df[features]

grid_df['pred_score'] = model.predict(grid_X)

# Top 10 추천
top_k = grid_df.sort_values('pred_score', ascending=False).head(10)

print(top_k)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001174 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1501
[LightGBM] [Info] Number of data points in the train set: 534978, number of used features: 6
[LightGBM] [Info] Start training from score 812.500663
             위도          경도  store_density_500m  cafe_density_500m  \
631   37.494655  127.030400                6958                172   
681   37.500005  127.030400                6135                208   
579   37.489305  127.013482                5476                121   
629   37.494655  127.013482                5406                105   
734   37.505355  127.055775                5211                144   
1275  37.564205  126.979648                4884                207   
1326  37.569555  126.988107                4647                222   
682   37.500005  127.038858     